# H1b 手先で指示する（逆運動学と回転）

**ねらい**：H1a では関節に角度で指示した。作品では「手先をこの位置へ、真下に向けて」と指示している。
手先の位置と向きから関節の角度を逆算するのが **逆運動学（IK）**。ここでは、
- 手先の **位置** の指示：どの点を動かしているのか（目印 `gripperframe`）
- 手先の **向き** の指示：真下に向ける回転の作り方
- **届く範囲**：真下向きのまま、どこまで届くか
- **解き直す間隔**：動きながら IK を何 step ごとに解き直すか
- **2 台目の向き**：180° 回して置いた腕では、回転の作り方を変えないと手先が横を向く（わざと失敗 `world_tilt`）

を、自分で値を書き換えて確かめる。

## 1. 準備

In [ ]:
import os, time
if "google.colab" in str(get_ipython()):
    !apt-get -qq install -y libosmesa6 fonts-noto-cjk > /dev/null
    !pip -q install genesis-world==1.4.3
    if not os.path.exists("SO-ARM100"):
        !git clone -q https://github.com/TheRobotStudio/SO-ARM100.git && cd SO-ARM100 && git checkout -q a758567
    MODEL = "/content/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
else:   # 手元の Docker（答え合わせ用）
    MODEL = "/work/third_party/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
import numpy as np, matplotlib.pyplot as plt, genesis as gs
# グラフの日本語：Colab は matplotlib を先に読み込んでいるので、入れたフォントを直接登録する（名前だけ指定しても見つからない）
from matplotlib import font_manager
for _f in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
    if os.path.exists(_f):
        font_manager.fontManager.addfont(_f)
plt.rcParams["font.family"] = ["Noto Sans CJK JP", "DejaVu Sans"]

## 2. 道具：回転を扱う関数

- `rot(q)`：クォータニオン（向きを表す 4 つの数 w, x, y, z）→ 回転の行列（3×3）
- `quat_of(R)`：回転の行列 → クォータニオン（IK に向きを渡すときの形）
- `Ry(a)`・`Rz(a)`：y 軸・z 軸のまわりに a [rad] 回す行列。**行列を左から掛けると、世界の軸のまわりに回す**
- `OFF`：手首の部品（`gripper` リンク）の原点から、**目印**（固定側の指の内側の先端、`gripperframe`）までのずれ。IK は部品の原点を動かすので、目印を目標に置くには、このずれを引いて指示する

In [ ]:
import math
OFF = np.array([-0.0079, -0.000218, -0.0981])        # gripper の原点 → 目印（gripper の向きで見た長さ [m]）
FOLD = np.array([0.0, -1.7, 1.6, 1.2, 0.0, 0.0])     # たたんだ姿勢（H0 で刺さった話を参照）
OPEN, CLOSE = 1.2, -0.17                             # 指の角度（開く／閉じきる）

def rot(q):
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])

def quat_of(R):
    """回転の行列 → クォータニオン。どの向きでも 0 で割らない形（作品の共通部品と同じ）。"""
    t = np.trace(R)
    if t > 0:
        S = math.sqrt(t + 1.0) * 2; q = [0.25 * S, (R[2, 1] - R[1, 2]) / S, (R[0, 2] - R[2, 0]) / S, (R[1, 0] - R[0, 1]) / S]
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        S = math.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2]) * 2; q = [(R[2, 1] - R[1, 2]) / S, 0.25 * S, (R[0, 1] + R[1, 0]) / S, (R[0, 2] + R[2, 0]) / S]
    elif R[1, 1] > R[2, 2]:
        S = math.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2]) * 2; q = [(R[0, 2] - R[2, 0]) / S, (R[0, 1] + R[1, 0]) / S, 0.25 * S, (R[1, 2] + R[2, 1]) / S]
    else:
        S = math.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1]) * 2; q = [(R[1, 0] - R[0, 1]) / S, (R[0, 2] + R[2, 0]) / S, (R[1, 2] + R[2, 1]) / S, 0.25 * S]
    q = np.array(q); return q / np.linalg.norm(q)

def Ry(a): return np.array([[math.cos(a), 0, math.sin(a)], [0, 1, 0], [-math.sin(a), 0, math.cos(a)]])
def Rz(a): return np.array([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0], [0, 0, 1]])

## 3. 2 台を置く

その 1 は原点に、その 2 は 44 cm 先に **180° 回して**（向かい合わせに）置く。赤い玉は「手先をここへ」の目標（ぶつからない飾り）。

In [ ]:
gs.init(backend=gs.cpu, logging_level="warning")
sc = gs.Scene(sim_options=gs.options.SimOptions(dt=0.01, substeps=10), show_viewer=False)
sc.add_entity(gs.morphs.Plane())
D = 0.44
A = sc.add_entity(gs.morphs.MJCF(file=MODEL, pos=(0, 0, 0), decompose_robot_error_threshold=0.0))
B = sc.add_entity(gs.morphs.MJCF(file=MODEL, pos=(D, 0, 0), euler=(0, 0, 180), decompose_robot_error_threshold=0.0))
goal = sc.add_entity(gs.morphs.Sphere(radius=0.008, pos=(0.22, 0, 0.06), fixed=True, collision=False),
                     surface=gs.surfaces.Default(color=(0.9, 0.1, 0.1)))
cam = sc.add_camera(res=(768, 432), pos=(0.22, -0.50, 0.28), lookat=(0.22, 0.0, 0.06), fov=50, GUI=False)
t0 = time.time(); sc.build(); print(f"build {time.time() - t0:.0f} s")
gA, gB = A.get_link("gripper"), B.get_link("gripper")

# 関節 0 の姿勢での手首の向き R0 を測る（真下向きの回転は、この向きを起点に作る）。測ったら、たたんだ姿勢へ
A.set_dofs_position(np.zeros(6)); B.set_dofs_position(np.zeros(6))
R0A, R0B = rot(gA.get_quat().cpu().numpy()), rot(gB.get_quat().cpu().numpy())
FINGER = R0A.T @ np.array([1.0, 0, 0])      # 手首の部品から見た「指の向き」（関節 0 の姿勢では、その 1 の指は世界の +x を向く）
for arm in (A, B):
    arm.set_dofs_position(FOLD); arm.control_dofs_position(FOLD)

def tip(arm, g):
    """目印（固定側の指の内側の先端）の位置。"""
    return g.get_pos().cpu().numpy() + rot(g.get_quat().cpu().numpy()) @ OFF

def tilt_from_down(g):
    """指の向きが、真下から何度ずれているか。"""
    d = rot(g.get_quat().cpu().numpy()) @ FINGER
    return math.degrees(math.acos(max(-1.0, min(1.0, -d[2]))))

def show():
    sc.visualizer.update(force=True)   # 瞬間移動（set_dofs_position）の後は、時間を進めないと絵が古いままなので、描く前に更新する
    plt.figure(figsize=(8, 4.5)); plt.imshow(cam.render(rgb=True)[0]); plt.axis("off"); plt.show()

print("R0（その 1）=\n", R0A.round(3))
show()

**期待される出力**：2 台ともたたんだ姿勢。R0 は、関節 0 の姿勢で手首の部品が世界に対してどう回っているか。

## 4. 手先を、位置と向きで指示する

真下向きの回転 `Rd = Ry(90°) @ R0`：関節 0 の姿勢（指は水平に前を向く）から、**世界の y 軸のまわりに 90° 倒す** と、指が真下を向く。
`inverse_kinematics(link=手首, pos=目標 − Rd @ OFF, quat=向き)` は、そうなる関節の角度を返す。
ここでは、返った角度に **瞬間移動**（`set_dofs_position`）して、目印が目標に来たか・真下を向いたかだけを確かめる（重さで垂れる話は 6 で）。

In [ ]:
def reach(arm, g, R0, base, yaw, r, z, tilt=math.pi / 2, world_axis=False):
    """arm の手先（目印）を、根元から見て前へ r [m]・高さ z [m] に、真下向き（tilt）で置く角度を IK で解き、そこへ瞬間移動する。
    world_axis=True：世界の y 軸のまわりに倒す（その 1 は同じ。180° 回したその 2 では違う）"""
    target = np.array([base[0] + r * math.cos(yaw), base[1] + r * math.sin(yaw), z])
    Rd = Ry(tilt) @ R0 if world_axis else Rz(yaw) @ Ry(tilt) @ Rz(-yaw) @ R0    # 腕自身の向きで見た横の軸のまわりに倒す
    arm.set_dofs_position(FOLD)
    q = arm.inverse_kinematics(link=g, pos=target - Rd @ OFF, quat=quat_of(Rd), init_qpos=arm.get_dofs_position(),
                               max_solver_iters=200).cpu().numpy()
    q[5] = OPEN
    arm.set_dofs_position(q)
    err = np.linalg.norm(tip(arm, g) - target) * 1000
    return q, err, tilt_from_down(g)

goal.set_pos(np.array([0.22, 0, 0.06]))
q, err, tl = reach(A, gA, R0A, (0, 0), 0.0, r=0.22, z=0.06)
print(f"関節の角度 {q.round(2)}   目印のずれ {err:.1f} mm   真下からのずれ {tl:.1f}°")
show()

**期待される出力**：目印のずれは数 mm 以下、真下からのずれは 1° 未満。窓の絵で、その 1 の指先が赤い玉に下りている。

## ✏️ 変えてみる 1：倒す角度
**予想**：`tilt` を 60°、120° にすると、指はどちらを向く？

In [ ]:
for deg in [60, 90, 120]:
    q, err, tl = reach(A, gA, R0A, (0, 0), 0.0, r=0.22, z=0.06, tilt=math.radians(deg))
    print(f"tilt {deg:3d}°   目印のずれ {err:5.1f} mm   真下からのずれ {tl:5.1f}°")

## ✏️ 変えてみる 2：届く範囲
真下向きのまま、根元からの距離 r と高さ z を振る。**手首（`wrist_flex`、3 番）が可動範囲の端（1.66 rad）に張り付いた** ら、それ以上は真下を向けない。
**予想**：届くのは、根元から何 cm〜何 cm？ 高い所と低い所、どちらが届きやすい？

In [ ]:
RS = [0.12, 0.16, 0.20, 0.24, 0.28, 0.32]
ZS = [0.02, 0.06, 0.10, 0.15]
lim = 1.658
print("目印のずれ [mm] / 真下からのずれ [°]（* は手首が可動範囲の端）")
print("z \\ r  " + "".join(f"{r * 100:>12.0f}cm" for r in RS))
for z in ZS:
    row = []
    for r in RS:
        q, err, tl = reach(A, gA, R0A, (0, 0), 0.0, r, z)
        row.append(f"{err:5.0f}/{tl:3.0f}{'*' if abs(q[3]) > lim - 0.02 else ' '}")
    print(f"{z * 100:3.0f}cm   " + "".join(f"{c:>14s}" for c in row))

**見るところ**：ずれが小さく（数 mm）真下（0°）のマスが「届く」。作品では **根元から 16〜28 cm・低い所** だけを使っている（台座 2 cm、運ぶ高さ 6〜7 cm）。

## 5. 2 台目：180° 回して置いた腕
その 2 で、同じ「真下に倒す」を 2 通りの作り方で解く。
- **腕自身の向きで倒す**：`Rz(yaw) @ Ry(90°) @ Rz(−yaw) @ R0`（いったん腕の向きを 0 に戻して倒し、また腕の向きへ回す）
- **世界の軸で倒す**：`Ry(90°) @ R0`（その 1 と同じ式。わざと失敗 `world_tilt`）

**予想**：世界の軸で倒すと、その 2 の指はどちらを向く？

In [ ]:
goal.set_pos(np.array([D - 0.22, 0, 0.06]))
A.set_dofs_position(FOLD)   # その 1 はたたんでおく（前の表の最後の姿勢が残っていると見づらい）
for world_axis in [False, True]:
    q, err, tl = reach(B, gB, R0B, (D, 0), math.pi, r=0.22, z=0.06, world_axis=world_axis)
    print(f"{'世界の軸で倒す' if world_axis else '腕自身の向きで倒す'}   目印のずれ {err:5.1f} mm   真下からのずれ {tl:5.1f}°")
    show()

**なぜ**：`Ry(90°)` を左から掛けると、**世界の** y 軸のまわりに回る。その 1（向き 0°）では、世界の y 軸が腕の横の軸と同じなので、指が前から下へ倒れる。
180° 回したその 2 では、R0B の指は世界の −x を向いている。世界の y 軸のまわりに +90° 回すと、−x を向いた指は **上** へ倒れる（または IK が届かない姿勢になり、横を向いて止まる）。

## 6. 動かしながら解き直す（実際の動き）
4・5 は瞬間移動だった。実際は、目標を S 字で少しずつ動かしながら、**5 step ごとに IK を解き直して** 関節に指示する（作品の共通部品 `move` と同じ）。
ここでは、掴みに行くときと同じ動き：真下向きで箱の上（高さ 8 cm）に置いてから、**高さ 2 cm（3 cm の箱の真ん中）まで下ろす**。
動かす間、指は **閉じておく**（開いたまま動かすと、向かいのその 2 の指に当たる。✏️ 4 と 📕 2）。

**予想**：下ろす時間を 1 秒・0.3 秒にすると、また目標を一気に置く（わざと失敗 `jump`）と、手先は 2 cm より下へ行き過ぎる？
（作品では、行き過ぎると箱の上に着地したり、箱を押しのけたりした）

In [ ]:
def descend(arm, g, R0, r=0.22, z_from=0.08, z_to=0.02, n=100, every=5, grip=CLOSE, jump=False):
    """真下向きで (r, z_from) に置いてから、n step かけて S 字で z_to まで下ろす。every step ごとに IK を解き直す。
    jump=True：目標を一度に置く（わざと失敗 jump）。手先の通り道を返す"""
    sc.reset()
    B.set_dofs_position(FOLD); B.control_dofs_position(FOLD)
    q, _, _ = reach(arm, g, R0, (0, 0), 0.0, r, z_from); q[5] = grip
    arm.set_dofs_position(q); arm.control_dofs_position(q)
    for _ in range(30):
        sc.step()
    start, target = tip(arm, g), np.array([r, 0.0, z_to])
    Rd = Ry(math.pi / 2) @ R0
    path = []
    for k in range(n + 50):
        s = min(1.0, (k + 1) / (0.7 * n)); s = 1.0 if jump else 3 * s ** 2 - 2 * s ** 3
        cur = start + (target - start) * s
        if k % every == 0:
            q = arm.inverse_kinematics(link=g, pos=cur - Rd @ OFF, quat=quat_of(Rd), init_qpos=arm.get_dofs_position(),
                                       max_solver_iters=50).cpu().numpy()
            q[5] = grip
            arm.control_dofs_position(q)
        sc.step()
        path.append(tip(arm, g))
    return np.array(path)

goal.set_pos(np.array([0.22, 0, 0.02]))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for label, kw in [("S 字で 1 秒", dict(n=100)), ("S 字で 0.3 秒", dict(n=30)), ("一気に（jump）", dict(n=100, jump=True))]:
    P = descend(A, gA, R0A, **kw)
    t_ = np.arange(len(P)) * 0.01
    ax[0].plot(t_, P[:, 2] * 100, label=label); ax[1].plot(t_, (P[:, 0] - 0.22) * 1000, label=label)
    print(f"{label:12s}  2 cm より下へ行き過ぎ {max(0.02 - P[:, 2].min(), 0) * 1000:4.1f} mm   横へのずれの最大 {np.abs(P[:, 0] - 0.22).max() * 1000:4.1f} mm   着いた所 {P[-1].round(3)}")
ax[0].axhline(2, color="gray", ls="--", lw=0.8); ax[0].set(xlabel="s", ylabel="目印の高さ [cm]"); ax[0].legend()
ax[1].set(xlabel="s", ylabel="横（前後）へのずれ [mm]"); plt.show()
show()

**期待される出力**（手元の実測）
```
S 字で 1 秒      2 cm より下へ行き過ぎ  0.5 mm   横へのずれの最大  6.7 mm
S 字で 0.3 秒    2 cm より下へ行き過ぎ  0.9 mm   横へのずれの最大  5.7 mm
一気に（jump）   2 cm より下へ行き過ぎ  9.3 mm   横へのずれの最大 12.1 mm
```
一気に置くと、目印が床から 1 cm まで下がる。箱があれば、箱の上の面に当たって押しのける。

## ✏️ 変えてみる 3：IK を解き直す間隔
**予想**：`every` を 1・25 にすると？（作品は 5。並列の環境が多いと IK の計算が重いので、間引いている）

In [ ]:
for every in [1, 5, 25]:
    t0 = time.time(); P = descend(A, gA, R0A, n=100, every=every)
    print(f"{every:2d} step ごと   2 cm より下へ行き過ぎ {max(0.02 - P[:, 2].min(), 0) * 1000:4.1f} mm   横へのずれの最大 {np.abs(P[:, 0] - 0.22).max() * 1000:4.1f} mm   計算 {time.time() - t0:4.1f} s")

## ✏️ 変えてみる 4：指を開いたまま、たたんだ姿勢から動かす
作品の初めと同じ、たたんだ姿勢から箱の上へ動かす。**予想**：指を開いたまま（`OPEN`）だと、着いたときのずれは？ 腕のモーターの力は？

In [ ]:
def from_fold(grip, target=np.array([0.22, 0, 0.06]), n=150):
    """たたんだ姿勢から、target へ真下向きで S 字に動かす。着いたときのずれ [mm] を返す"""
    sc.reset()
    for a in (A, B):
        a.set_dofs_position(FOLD); a.control_dofs_position(FOLD)
    for _ in range(30):
        sc.step()
    start, Rd = tip(A, gA), Ry(math.pi / 2) @ R0A
    for k in range(n + 50):
        s = min(1.0, (k + 1) / (0.7 * n)); s = 3 * s ** 2 - 2 * s ** 3
        if k % 5 == 0:
            q = A.inverse_kinematics(link=gA, pos=start + (target - start) * s - Rd @ OFF, quat=quat_of(Rd),
                                     init_qpos=A.get_dofs_position(), max_solver_iters=50).cpu().numpy()
            q[5] = grip
            A.control_dofs_position(q)
        sc.step()
    return np.linalg.norm(tip(A, gA) - target) * 1000

goal.set_pos(np.array([0.22, 0, 0.06]))
for grip, label in [(CLOSE, "指を閉じて"), (OPEN, "指を開いたまま")]:
    e = from_fold(grip)
    print(f"{label:8s}  着いたときのずれ {e:5.1f} mm   腕のモーターの力 {A.get_dofs_control_force().cpu().numpy()[:4].round(2)} N·m")
c = A.get_contacts(with_entity=B)
print("その 2 と触れている部品の番号:", sorted({int(x) for x in c["link_a"].cpu().numpy().ravel()} | {int(x) for x in c["link_b"].cpu().numpy().ravel()}))
print("（その 1 の部品の番号:", [(l.idx, l.name) for l in A.links][-2:], " その 2:", [(l.idx, l.name) for l in B.links][-2:], "）")
show()

## 📕 作者の失敗の記録：2 台が刺さっていた（2026-10-20）
- **何が起きたか**：H0 の確認の絵で、2 台の手先が真ん中で重なっていた（作者が気づいた）
- **気づいた数字**：エラーは出ない。シミュレーション 1 秒ぶんの計算が 1.2 秒かかっていた
- **切り分け**：組み立てた直後の姿勢を見ると、関節が全部 0（前腕を水平に伸ばす）。手先は根元から 22 cm 以上。2 台を 44 cm 離して向かい合わせると、真ん中で重なる
- **原因**：初めの姿勢を決めていなかった
- **対策**：組み立てた直後に、角度とモーターの目標の両方をたたんだ姿勢 `FOLD` にする（角度だけだと、次の step から伸びに行く）
- **学び**：計算が 1.2 秒 → 0.15 秒になった。**ぶつかり続けている物は、エラーを出さずに計算だけを重くする**。絵で確かめる

## 📕 作者の失敗の記録 2：条件を変えても、結果が同じだった（2026-10-20）
- **何が起きたか**：6 を最初に書いたとき（たたんだ姿勢から動かし、IK を解き直す間隔を 1・5・25 step に振った）、どれでも「着いたときのずれ 50.2 mm」で同じだった
- **気づいた数字**：肩・肘・手首のモーターの力が、3 つとも力の上限（±3.35 N·m）に張り付いていた。何かを押している
- **切り分け**：触れている部品を `get_contacts` で調べると、その 1 の動く側の指（`moving_jaw`）と、その 2 の動く側の指。その 2 を横へ向けても、その 1 の指を閉じても、ずれは 0.5 mm
- **原因**：指を開いたまま動かし、開いた指が、向かいのその 2 のたたんだ指に当たっていた
- **対策**：動かす間は指を閉じる（作品でも、運ぶ途中の指は閉じるか半分だけ開く。その 2 は離れた場所で待つ）
- **学び**：H1a の床と同じ形。**結果が条件によらず同じなら、何かに当たっている**。モーターの力が上限に張り付いていないかを見る
- **もう 1 つ**：たたんだ姿勢から動かすと、手先は真下を向いておらず低い所（1.2 cm）から始まる。動き出しで手首が大きく振れ、「下への行き過ぎ」が初めの高さを拾っていた。演習は、真下向きで箱の上に置いてから下ろす形に変えた

## 確かめる問い
1. IK に `pos=目標 − Rd @ OFF` を渡すのはなぜ？
2. その 1 では `Ry(90°) @ R0` で真下を向くのに、その 2 では向かないのはなぜ？
3. 作品で、運ぶ高さを 8 cm 以上にしないのはなぜ？

<details><summary>答え</summary>

1. IK が動かすのは手首の部品の原点で、目標に置きたいのは指先の目印。部品の向き Rd で見たずれ OFF の分だけ、手前に指示する
2. 行列を左から掛けると世界の軸のまわりに回る。その 1 は腕の横の軸が世界の y 軸と同じだが、180° 回したその 2 は、指が世界の −x を向いているので、同じ回し方では反対（上）へ倒れる
3. 根元から 20 cm で 8 cm 以上に上げると、手首が可動範囲の端に張り付いて、真下を向けきれない（✏️ 2 の表）
</details>